Requirements: `torch`; `torchvision`.

In [1]:
# MacOS bug
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import StepLR

In [3]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)
        return output

In [4]:
def train(model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 100 == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))


def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))

In [5]:
batch_size = 64  # input batch size for training
test_batch_size = 1000  # input batch size for testing
epochs = 2  # number of epochs to train
lr = 1.0  # learning rate
gamma = 0.7  # Learning rate step gamma

In [6]:
no_accel = True  # disables accelerator

use_accel = not no_accel and torch.accelerator.is_available()

if use_accel:
    device = torch.accelerator.current_accelerator()
else:
    device = torch.device("cpu")

In [ ]:
train_kwargs = {'batch_size': batch_size}
test_kwargs = {'batch_size': test_batch_size}
if use_accel:
    accel_kwargs = {'num_workers': 1,
                    'persistent_workers': True,
                    'pin_memory': True,
                    'shuffle': True}
    train_kwargs.update(accel_kwargs)
    test_kwargs.update(accel_kwargs)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
dataset1 = datasets.MNIST('./data', train=True, download=True, transform=transform)
dataset2 = datasets.MNIST('./data', train=False, transform=transform)
train_loader = torch.utils.data.DataLoader(dataset1, **train_kwargs)
test_loader = torch.utils.data.DataLoader(dataset2, **test_kwargs)

model = Net().to(device)

optimizer = optim.Adadelta(model.parameters(), lr=lr)
scheduler = StepLR(optimizer, step_size=1, gamma=gamma)

for epoch in range(1, epochs + 1):
    train(model, device, train_loader, optimizer, epoch)
    test(model, device, test_loader)
    scheduler.step()

Train Epoch: 1 [0/60000 (0%)]	Loss: 2.308417
Train Epoch: 1 [6400/60000 (11%)]	Loss: 0.318414
Train Epoch: 1 [12800/60000 (21%)]	Loss: 0.099559
Train Epoch: 1 [19200/60000 (32%)]	Loss: 0.176271
Train Epoch: 1 [25600/60000 (43%)]	Loss: 0.049202
Train Epoch: 1 [32000/60000 (53%)]	Loss: 0.146862
Train Epoch: 1 [38400/60000 (64%)]	Loss: 0.175837
Train Epoch: 1 [44800/60000 (75%)]	Loss: 0.152793
Train Epoch: 1 [51200/60000 (85%)]	Loss: 0.231684
Train Epoch: 1 [57600/60000 (96%)]	Loss: 0.157172

Test set: Average loss: 0.0502, Accuracy: 9831/10000 (98%)

Train Epoch: 2 [0/60000 (0%)]	Loss: 0.095393
Train Epoch: 2 [6400/60000 (11%)]	Loss: 0.163442
Train Epoch: 2 [12800/60000 (21%)]	Loss: 0.077193
Train Epoch: 2 [19200/60000 (32%)]	Loss: 0.023618
Train Epoch: 2 [25600/60000 (43%)]	Loss: 0.100072
Train Epoch: 2 [32000/60000 (53%)]	Loss: 0.039662
Train Epoch: 2 [38400/60000 (64%)]	Loss: 0.109963
Train Epoch: 2 [44800/60000 (75%)]	Loss: 0.153188
Train Epoch: 2 [51200/60000 (85%)]	Loss: 0.169195
T